# 📉 Analyse de Sensibilité : Seuil Safety vs F1 Score

Ce notebook visualise l'impact du seuil de sécurité (`safety_check`) sur la performance du Syndicat.

**Attention :** N'ayant pas les vrais labels du Test Set, nous utilisons ici **Sénat V9** comme "Cible de Référence" pour calculer les scores. 
*Cela mesure donc "À quel point le Syndicat ressemble au Sénat".*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder

# 1. Re-Load Data needed for Proba
train_df = pd.read_csv('conversion_data_train.csv')
test_df = pd.read_csv('conversion_data_test.csv')
v9_labels = pd.read_csv('submission_LE_SENAT_V9.csv')
test_df['converted'] = v9_labels['converted']
full_df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

# Feature Eng (Fast Version)
def feature_engineering(df):
    df = df.copy()
    df['pages_per_age'] = df['total_pages_visited'] / (df['age'] + 0.1)
    df['interaction_age_pages'] = df['age'] * df['total_pages_visited']
    df['is_active'] = (df['total_pages_visited'] > 2).astype(int)
    return df

X_full = feature_engineering(full_df.drop('converted', axis=1))
y_full = full_df['converted']
X_test = feature_engineering(test_df.drop('converted', axis=1))

cat_cols = ['country', 'source']
cat_indices = []
for col in cat_cols:
    le = LabelEncoder()
    le.fit(X_full[col])
    X_full[col] = le.transform(X_full[col])
    X_test[col] = le.transform(X_test[col])
    cat_indices.append(X_full.columns.get_loc(col))

# Re-Train Monster quickly to get final_proba
model = HistGradientBoostingClassifier(
    loss="log_loss", learning_rate=0.03, max_iter=500, max_depth=8,
    l2_regularization=0.1, categorical_features=cat_indices, random_state=42
)
model.fit(X_full, y_full)
final_proba = model.predict_proba(X_test)[:, 1]

# Load Others
sniper_pred = pd.read_csv('submission_FN_SNIPER.csv')['converted']
senate_yes = v9_labels['converted']

In [ ]:
# 2. Visualization Logic (User Provided + Adapted)
thresholds = np.linspace(0.01, 0.50, 50)  # Extended Range
f1_scores = []
precisions = []
recalls = []

monster_yes = (final_proba >= 0.5).astype(int)

# Reference is SENATE V9 (As per request)
y_true_proxy = senate_yes

for t in thresholds:
    safety_check = (final_proba >= t).astype(int)
    syndicate_pred = (monster_yes | sniper_pred | (senate_yes & safety_check)).astype(int)
    
    precisions.append(precision_score(y_true_proxy, syndicate_pred))
    recalls.append(recall_score(y_true_proxy, syndicate_pred))
    f1_scores.append(f1_score(y_true_proxy, syndicate_pred))

plt.figure(figsize=(10,6))
plt.plot(thresholds, f1_scores, label='F1 Score (vs Senate)', color='purple', marker='o')
plt.plot(thresholds, precisions, label='Precision (vs Senate)', color='green', linestyle='--')
plt.plot(thresholds, recalls, label='Recall (vs Senate)', color='blue', linestyle='--')

plt.axvline(0.15, color='orange', linestyle='-', label='Optimal Threshold (0.15)')
plt.axhline(0.77, color='red', linestyle=':', label='Target F1 0.77')

plt.xlabel('Safety Threshold (Monster Proba)')
plt.ylabel('Score (Similarity to Senate)')
plt.title('Impact of Safety Threshold on Convergence to Senate V9')
plt.legend()
plt.grid(True)
plt.show()